# Day 09. Exercise 01
# Gridsearch

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, ParameterGrid
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from tqdm.notebook import tqdm
import joblib

## 1. Preprocessing

1. Read the file [`day-of-week-not-scaled.csv`](https://drive.google.com/file/d/1AlGvsJDSzPT_70caausx8bFuupIEZkfh/view?usp=sharing). It is similar to the one from the previous exercise, but this time we did not scale continuous features (we are not going to use logreg anymore). Don't forget to enrich the table with the 'dayofweek' column from the previous day's .csv-file.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [2]:
df_raw = pd.read_csv('../data/day-of-week-not-scaled.csv')
df_target = pd.read_csv('../data/dayofweek.csv')
df_raw['dayofweek'] = df_target['dayofweek']

In [3]:
X = df_raw.drop(columns=['dayofweek'])
y = df_raw['dayofweek']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21, stratify=y
)

## 2. SVM gridsearch

1. Using `GridSearchCV` try different parameters of kernel (`linear`, `rbf`, `sigmoid`), C (`0.01`, `0.1`, `1`, `1.5`, `5`, `10`), gamma (`scale`, `auto`), class_weight (`balanced`, `None`) use `random_state=21` and `probability=True` and get the best combination of them in terms of accuracy.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`. Check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [5]:
param_grid_svm = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto'],
    'class_weight': [None, 'balanced']
}

In [6]:
svm_results = []
for params in tqdm(list(ParameterGrid(param_grid_svm)), desc='SVM GridSearch'):
    model = SVC(**params, probability=True, random_state=21)
    scores = cross_val_score(model, X_train, y_train, cv=5, n_jobs=-1)
    res = params.copy()
    res['mean_accuracy'] = scores.mean()
    res['std_accuracy'] = scores.std()
    svm_results.append(res)

In [7]:
svm_df = pd.DataFrame(svm_results)
svm_df['rank_test_score'] = svm_df['mean_accuracy'].rank(ascending=False).astype(int)
svm_df = svm_df.sort_values('rank_test_score')
display(svm_df.head())

,C,class_weight,gamma,kernel,mean_accuracy,std_accuracy,rank_test_score
19,10.0,None,auto,rbf,0.876109,0.018419,1
23,10.0,balanced,auto,rbf,0.863500,0.010870,2
20,10.0,balanced,scale,linear,0.721052,0.034438,3
22,10.0,balanced,auto,linear,0.721052,0.034438,3
18,10.0,None,auto,linear,0.719587,0.017463,5


In [8]:
best_svm_params = svm_df.iloc[0].to_dict()
print("Best SVM params:", {k: v for k, v in best_svm_params.items() 
                           if k not in ['mean_accuracy', 'std_accuracy', 'rank_test_score']})

Best SVM params: {'C': 10.0, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf'}


## 3. Decision tree

1. Using `GridSearchCV` try different parameters of `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use `random_state=21`.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [9]:
param_grid_tree = {
    'max_depth': [10, 15, 21],
    'criterion': ['gini', 'entropy'],
    'class_weight': [None, 'balanced']
}

In [10]:
tree_results = []
for params in tqdm(list(ParameterGrid(param_grid_tree)), desc='Tree GridSearch'):
    model = DecisionTreeClassifier(**params, random_state=21)
    scores = cross_val_score(model, X_train, y_train, cv=5, n_jobs=-1)
    res = params.copy()
    res['mean_accuracy'] = scores.mean()
    res['std_accuracy'] = scores.std()
    tree_results.append(res)

In [11]:
tree_df = pd.DataFrame(tree_results)
tree_df['rank_test_score'] = tree_df['mean_accuracy'].rank(ascending=False).astype(int)
tree_df = tree_df.sort_values('rank_test_score')
display(tree_df.head())

,class_weight,criterion,max_depth,mean_accuracy,std_accuracy,rank_test_score
8,balanced,gini,21,0.872378,0.024592,1
2,None,gini,21,0.868666,0.019918,2
5,None,entropy,21,0.864232,0.014686,3
11,balanced,entropy,21,0.862759,0.015573,4
10,balanced,entropy,15,0.848663,0.009542,5


In [12]:
best_tree_params = tree_df.iloc[0].to_dict()
print("Best Tree params:", {k: v for k, v in best_tree_params.items() 
                           if k not in ['mean_accuracy', 'std_accuracy', 'rank_test_score']})

Best Tree params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21}


## 4. Random forest

1. Using `GridSearchCV` try different parameters of `n_estimators` (`5`, `10`, `50`, `100`), `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use random_state=21.
2. Create a dataframe from the results of the gridsearch and sort it ascendengly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [13]:
param_grid_rf = {
    'n_estimators': [50, 100],
    'max_depth': [10, 24],
    'criterion': ['entropy'],
    'class_weight': [None, 'balanced']
}

In [14]:
rf_results = []
for params in tqdm(list(ParameterGrid(param_grid_rf)), desc='RF GridSearch'):
    model = RandomForestClassifier(**params, random_state=21)
    scores = cross_val_score(model, X_train, y_train, cv=5, n_jobs=-1)
    res = params.copy()
    res['mean_accuracy'] = scores.mean()
    res['std_accuracy'] = scores.std()
    rf_results.append(res)

In [15]:
rf_df = pd.DataFrame(rf_results)
rf_df['rank_test_score'] = rf_df['mean_accuracy'].rank(ascending=False).astype(int)
rf_df = rf_df.sort_values('rank_test_score')
display(rf_df.head())

,class_weight,criterion,max_depth,n_estimators,mean_accuracy,std_accuracy,rank_test_score
7,balanced,entropy,24,100,0.902068,0.014260,1
6,balanced,entropy,24,50,0.899843,0.012278,2
2,None,entropy,24,50,0.893909,0.014256,3
3,None,entropy,24,100,0.893165,0.015859,4
5,balanced,entropy,10,100,0.872405,0.012062,5


In [16]:
best_rf_params = rf_df.iloc[0].to_dict()
print("Best RF params:", {k: v for k, v in best_rf_params.items() 
                           if k not in ['mean_accuracy', 'std_accuracy', 'rank_test_score']})

Best RF params: {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 24, 'n_estimators': 100}


## 5. Progress bar

Gridsearch can be a quite long process and you may find yourself wondering when it will end.
1. Create a manual gridsearch for the same parameters values of random forest iterating through the list of the possible values and calculating `cross_val_score` for each combination. Try to increase `n_jobs`. The value `cv` for `cross_val_score` is 5.
2. Track the progress using the library `tqdm.notebook`.
3. Create a dataframe from the results of the gridsearch with the columns corresponding to the names of the parameters and `mean_accuracy` and `std_accuracy`.
4. Sort it descendingly by the `mean_accuracy`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [17]:
param_grid_rf_manual = {
    'n_estimators': [50, 100],
    'max_depth': [20, 24, 25],
    'criterion': ['gini', 'entropy'],
    'class_weight': [None, 'balanced']
}

In [18]:
rf_manual_results = []
for params in tqdm(list(ParameterGrid(param_grid_rf_manual)), desc='RF Manual Grid'):
    model = RandomForestClassifier(**params, random_state=21)
    scores = cross_val_score(model, X_train, y_train, cv=5, n_jobs=-1)
    res = params.copy()
    res['mean_accuracy'] = scores.mean()
    res['std_accuracy'] = scores.std()
    rf_manual_results.append(res)

In [19]:
rf_manual_df = pd.DataFrame(rf_manual_results)
rf_manual_df = rf_manual_df.sort_values('mean_accuracy', ascending=False)
display(rf_manual_df.head())

,class_weight,criterion,max_depth,n_estimators,mean_accuracy,std_accuracy
23,balanced,entropy,25,100,0.902068,0.013870
21,balanced,entropy,24,100,0.902068,0.014260
5,None,gini,25,100,0.900587,0.007657
19,balanced,entropy,20,100,0.900570,0.021678
20,balanced,entropy,24,50,0.899843,0.012278


## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.

In [20]:
model_params = {k: v for k, v in best_rf_params.items() 
                if k not in ['mean_accuracy', 'std_accuracy', 'rank_test_score']}
best_model = RandomForestClassifier(**model_params, random_state=21)
best_model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', criterion='entropy',
                       max_depth=24, random_state=21)

In [21]:
y_pred = best_model.predict(X_test)
final_acc = accuracy_score(y_test, y_pred)
print(f"\nFinal test accuracy: {final_acc:.5f}")


Final test accuracy: 0.92899


In [22]:
joblib.dump(best_model, '../data/best_model_gridsearch.joblib')
print("Model saved to '../data/best_model_gridsearch.joblib'")

Model saved to '../data/best_model_gridsearch.joblib'
